In [1]:
import pandas as pd
import os

# ============================================================
# 0. CREATE OUTPUT FOLDER IF NOT EXISTS
# ============================================================

OUTPUT_DIR = "../data/datasets_hierarchical"
print("Saving all datasets to:", OUTPUT_DIR)

# ============================================================
# 1. SCENARIO DEFINITIONS
# ============================================================

# Natural groups
faults = list(range(1, 7))          # 1–6
maintenance = [13, 14]              # 13–14
normal = [41]                       # 41

# Attack groups
data_injection = list(range(7, 13))     # 7–12
remote_tripping = list(range(15, 21))   # 15–20
relay_setting = list(range(21, 31)) + list(range(35, 41))  # 21–30, 35–40

all_attacks = data_injection + remote_tripping + relay_setting
all_non_attacks = faults + maintenance + normal


# ============================================================
# 2. LOAD ORIGINAL DATASET
# ============================================================

df = pd.read_csv("../data/merged/multi_class_dataset_clean.csv")   # <-- CHANGE IF NEEDED
print("Loaded dataset shape:", df.shape)


# ============================================================
# 3. M1 – ATTACK VS NON-ATTACK
# ============================================================

df_m1 = df.copy()

df_m1["label"] = df_m1["marker"].apply(lambda s: 1 if s in all_attacks else 0)

df_m1["label_name"] = df_m1["label"].map({
    0: "Non-Attack",
    1: "Attack"
})
df_m1.to_csv(f"{OUTPUT_DIR}/M1_Attack_vs_NonAttack.csv", index=False)
print("Saved M1_Attack_vs_NonAttack.csv")


# ============================================================
# 4. M2 – NATURAL 3-CLASS
# ============================================================

df_m2 = df[df["marker"].isin(all_non_attacks)].copy()

def map_natural(s):
    if s in faults:
        return 0
    elif s in maintenance:
        return 1
    else:
        return 2

df_m2["label"] = df_m2["marker"].apply(map_natural)

df_m2["label_name"] = df_m2["label"].map({
    0: "Short-Circuit Fault",
    1: "Line Maintenance",
    2: "Normal Operaton"
})
df_m2.to_csv(f"{OUTPUT_DIR}/M2_Natural_3Class.csv", index=False)
print("Saved M2_Natural_3Class.csv")


# ============================================================
# 5. M3 – ATTACK 3-CLASS
# ============================================================

df_m3 = df[df["marker"].isin(all_attacks)].copy()

def map_attack(s):
    if s in data_injection:
        return 0
    elif s in remote_tripping:
        return 1
    else:
        return 2

df_m3["label"] = df_m3["marker"].apply(map_attack)

df_m3["label_name"] = df_m3["label"].map({
    0: "Data Injection",
    1: "Remote Tripping",
    2: "Relay Setting Change"
})
df_m3.to_csv(f"{OUTPUT_DIR}/M3_Attack_3Class.csv", index=False)
print("Saved M3_Attack_3Class.csv")


# ============================================================
# 6. M4 – DATA INJECTION INTERNAL (7–12)
# ============================================================

df_m4 = df[df["marker"].isin(data_injection)].copy()
df_m4["label"] = df_m4["marker"]

df_m4.to_csv(f"{OUTPUT_DIR}/M4_DataInjection_Internal.csv", index=False)
print("Saved M4_DataInjection_Internal.csv")


# ============================================================
# 7. M5 – REMOTE TRIPPING INTERNAL (15–20)
# ============================================================

df_m5 = df[df["marker"].isin(remote_tripping)].copy()
df_m5["label"] = df_m5["marker"]

df_m5.to_csv(f"{OUTPUT_DIR}/M5_RemoteTripping_Internal.csv", index=False)
print("Saved M5_RemoteTripping_Internal.csv")


# ============================================================
# 8. M6 – RELAY SETTING INTERNAL (21–30, 35–40)
# ============================================================

df_m6 = df[df["marker"].isin(relay_setting)].copy()
df_m6["label"] = df_m6["marker"]

df_m6.to_csv(f"{OUTPUT_DIR}/M6_RelaySetting_Internal.csv", index=False)
print("Saved M6_RelaySetting_Internal.csv")


Saving all datasets to: ../data/datasets_hierarchical
Loaded dataset shape: (78377, 129)
Saved M1_Attack_vs_NonAttack.csv
Saved M2_Natural_3Class.csv
Saved M3_Attack_3Class.csv
Saved M4_DataInjection_Internal.csv
Saved M5_RemoteTripping_Internal.csv
Saved M6_RelaySetting_Internal.csv


In [11]:
# -------------------------------
# 2. HELPER FUNCTIONS
# -------------------------------

def eda_basic(df, target_num="label", target_name="label_name"):
    """
    Performs Step 1–3 of EDA:
    1. Quick overview (info + summary stats)
    2. Class distribution (counts, percentages + plot)
    3. Correlation with numeric target + top-10 barplot
    Returns: list of top 10 correlated features
    """

    print("\n==============================")
    print("1. DATASET OVERVIEW")
    print("==============================")
    print("Shape:", df.shape)

    print("\nInfo:")
    print(df.info())

    print("\nSummary statistics:")
    print(df.describe().T.head(10))

    print("\n==============================")
    print("2. CLASS DISTRIBUTION")
    print("==============================")
    print("\nClass counts:")
    print(df[target_num].value_counts())

    print("\nClass percentages (%):")
    print(df[target_num].value_counts(normalize=True) * 100)

    # Plot class distribution
    plt.figure(figsize=(6,4))
    sns.countplot(x=target_name, data=df, palette="Set2")
    plt.title("Class Distribution")
    plt.tight_layout()
    plt.show()

    print("\n==============================")
    print("3. CORRELATION WITH TARGET")
    print("==============================")

    # Select numeric columns (excluding target_num)
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if target_num in numeric_cols:
        numeric_cols.remove(target_num)

    # Correlation with numeric target
    corr_series = df[numeric_cols].corrwith(df[target_num])
    corr_abs = corr_series.abs().sort_values(ascending=False)

    top10_features = corr_abs.head(10).index.tolist()

    print("\nTop 10 correlated features:")
    print(top10_features)

    # Barplot for correlations
    plt.figure(figsize=(12,6))
    ax = sns.barplot(
        x=top10_features,
        y=corr_abs[top10_features].values,
        color="#4C72B0"
    )
    for i, v in enumerate(corr_abs[top10_features].values):
        ax.text(i, v + 0.001, f"{v:.3f}", ha='center', va='bottom')
    plt.xticks(rotation=45, ha='right')
    plt.title("Top 10 Features Correlated With Target")
    plt.ylabel("Correlation (absolute)")
    plt.tight_layout()
    plt.show()

    return top10_features

In [12]:
df_m1 = pd.read_csv("../data/datasets_hierarchical/M1_Attack_vs_NonAttack.csv")
top10_m1 = eda_basic(df_m1)


1. DATASET OVERVIEW
Shape: (78377, 131)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78377 entries, 0 to 78376
Columns: 131 entries, R1-PA1:VH to label_name
dtypes: float64(128), int64(2), object(1)
memory usage: 78.3+ MB
None

Summary statistics:


/Users/cherylchau/Library/Mobile Documents/com~apple~CloudDocs/FYP-data/myenv/lib/python3.9/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/cherylchau/Library/Mobile Documents/com~apple~CloudDocs/FYP-data/myenv/lib/python3.9/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/cherylchau/Library/Mobile Documents/com~apple~CloudDocs/FYP-data/myenv/lib/python3.9/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/cherylchau/Library/Mobile Documents/com~apple~CloudDocs/FYP-data/myenv/lib/python3.9/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


             count           mean          std            min            25%  \
R1-PA1:VH  78377.0     -15.802424   100.876750    -179.988962    -100.416583   
R1-PM1:V   78377.0  131670.634209  1039.382656  129365.536650  131057.982300   
R1-PA2:VH  78377.0       2.175196   111.743169    -179.994691    -102.129727   
R1-PM2:V   78377.0  131361.012902  1070.855517  129001.974200  130732.029800   
R1-PA3:VH  78377.0       6.834315    97.065063    -179.994691     -69.459674   
R1-PM3:V   78377.0  131736.200705  1038.943651  129440.756300  131133.202100   
R1-PA4:IH  78377.0     -14.334996    99.601107    -179.994691     -98.159129   
R1-PM4:I   78377.0     380.592779   119.435979      79.469740     305.793700   
R1-PA5:IH  78377.0       3.538540   109.504977    -179.994691     -94.790138   
R1-PM5:I   78377.0     383.504558   115.162266      89.083015     311.836330   

                     50%            75%            max  
R1-PA1:VH     -28.865614      68.096034     179.994691  
R1-PM

NameError: name 'plt' is not defined